A Python notebook to check specific unique entity ratio (UER) per type between two datasets.

## Libraries and Data

In [2]:
import numpy as np
import pandas as pd
import math
from collections import Counter
from EXPERIMENTS.evaluate_gold import load_and_merge_gold_data
from dataset_processing import generate_consistent_label_map, CLIRENER_LABELS_V1, calculate_advanced_metrics

#### Helper functions

In [3]:

def get_filtered_dataset_for_type(dataset, target_type, bio_label_list):
    """
    Creates a copy of the dataset where only the target_type 
    retains its BIO tags; all other entities are masked to 'O'.
    """
    filtered_data = []
    # Identify which IDs in the bio_label_list belong to our target_type
    target_ids = [i for i, name in enumerate(bio_label_list) 
                  if name == f"B-{target_type}" or name == f"I-{target_type}"]
    
    for row in dataset:
        new_tags = [tag if tag in target_ids else 0 for tag in row['ner_tags']]
        filtered_data.append({
            'tokens': row['tokens'],
            'ner_tags': new_tags
        })
    return filtered_data

#### Data

In [4]:



# --- 2. Setup and Loading ---
DATA_ID_1 = "P0L3/CliReNER_v_1_1_28_GOLD"
DATA_ID_2 = "P0L3/CliReNER_v_1_1_28_SILVER"

# actual_types: ['Ecosystem', 'Location', ...]
# bio_labels: ['O', 'B-Ecosystem', 'I-Ecosystem', ...]
actual_types = CLIRENER_LABELS_V1
bio_labels = list(generate_consistent_label_map(CLIRENER_LABELS_V1))

data_1 = load_and_merge_gold_data(DATA_ID_1)
data_2 = load_and_merge_gold_data(DATA_ID_2)

# Using the test splits as requested
gold_test = data_1[0]["test"]
silver_test = data_2[0]["test"]

--- Loading and Merging GOLD Dataset: P0L3/CliReNER_v_1_1_28_GOLD ---
Merged ['train', 'validation', 'test'] into a single dataset of size: 192
--- Loading and Merging GOLD Dataset: P0L3/CliReNER_v_1_1_28_SILVER ---
Merged ['train', 'validation', 'test'] into a single dataset of size: 1027


## Calc

In [5]:
# --- 3. Systematic Execution ---
all_type_results = []

for entity_type in actual_types:
    # Filter datasets to isolate this specific entity type
    gold_filtered = get_filtered_dataset_for_type(gold_test, entity_type, bio_labels)
    silver_filtered = get_filtered_dataset_for_type(silver_test, entity_type, bio_labels)
    
    # Run your advanced metrics function
    gold_metrics = calculate_advanced_metrics(gold_filtered, bio_labels)
    silver_metrics = calculate_advanced_metrics(silver_filtered, bio_labels)
    
    if gold_metrics and silver_metrics:
        # We also need to calculate the "Novelty Gap" manually 
        # because the function looks at datasets in isolation.
        
        def get_vocab(dataset_split, target_type, label_names):
            # Helper to get the set of unique lowercased strings for overlap check
            vocab = set()
            for row in dataset_split:
                tokens, tags = row['tokens'], row['ner_tags']
                for i, tid in enumerate(tags):
                    if label_names[tid] == f"B-{target_type}":
                        j = i + 1
                        while j < len(tags) and label_names[tags[j]] == f"I-{target_type}":
                            j += 1
                        vocab.add(" ".join(tokens[i:j]).lower())
            return vocab

        g_vocab = get_vocab(gold_test, entity_type, bio_labels)
        s_vocab = get_vocab(silver_test, entity_type, bio_labels)
        
        overlap = len(g_vocab.intersection(s_vocab))
        novelty_rate = (1 - (overlap / len(g_vocab))) * 100 if g_vocab else 0

        all_type_results.append({
            "Type": entity_type,
            "Gold_Count": gold_metrics["Total Entities"],
            "Silver_Count": silver_metrics["Total Entities"],
            "Gold_TTR (UER)": round(gold_metrics["Entity TTR (0-1)"], 3),
            "Silver_TTR (UER)": round(silver_metrics["Entity TTR (0-1)"], 3),
            "TTR_Diff": round(gold_metrics["Entity TTR (0-1)"] - silver_metrics["Entity TTR (0-1)"], 3),
            "Novelty_Rate (%)": round(novelty_rate, 1),
            "Gold_Avg_Len": round(gold_metrics["Avg Span Len"], 2),
            "Gold_Clumping": round(gold_metrics["Clumping (Adj) %"], 1)
        })

# --- 4. Final Comparison Table ---
df = pd.DataFrame(all_type_results).sort_values("Novelty_Rate (%)", ascending=False)

print("\nCross-Type Comparison using Advanced Metrics")
print("=" * 100)
print(df.to_string(index=False))

# Outlier Detection for Ecosystem
eco_row = df[df["Type"] == "Ecosystem"]
avg_novelty = df["Novelty_Rate (%)"].mean()
avg_ttr_diff = df["TTR_Diff"].mean()

print("\n--- Diagnostic Findings ---")
if not eco_row.empty:
    e_nov = eco_row["Novelty_Rate (%)"].values[0]
    e_diff = eco_row["TTR_Diff"].values[0]
    
    print(f"Ecosystem Novelty Rate: {e_nov}% (Dataset Avg: {avg_novelty:.1f}%)")
    print(f"Ecosystem TTR Increase: {e_diff} (Dataset Avg: {avg_ttr_diff:.3f})")
    
    if e_nov > avg_novelty + 15:
        print(">> Ecosystem is an outlier in NOVELTY: The model is seeing words it never practiced.")
    if e_diff > 0.1:
        print(">> Ecosystem is an outlier in DIVERSITY: The silver set is too repetitive compared to gold.")


Cross-Type Comparison using Advanced Metrics
                     Type  Gold_Count  Silver_Count  Gold_TTR (UER)  Silver_TTR (UER)  TTR_Diff  Novelty_Rate (%)  Gold_Avg_Len  Gold_Clumping
       Natural Phenomenon          80            86           0.938             0.930     0.007              94.7          1.85           18.8
                   Policy          73            74           0.973             0.878     0.094              94.4          2.44           32.9
      Physical Phenomenon          72           278           0.944             0.665     0.279              91.2          1.65           15.3
        Physical Artefact          66            52           0.924             0.846     0.078              90.2          1.55           21.2
     Geographical Feature          57            89           0.965             0.888     0.077              89.1          1.86           12.3
  Mathematical Expression          45            62           0.867             0.903    -0.037 

In [10]:
df.sort_values(by="TTR_Diff", ascending=False)

,Type,Gold_Count,Silver_Count,Gold_TTR (UER),Silver_TTR (UER),TTR_Diff,Novelty_Rate (%),Gold_Avg_Len,Gold_Clumping
19,Meteorological Phenomenon,81,255,0.728,0.427,0.301,83.1,1.69,18.5
17,Chemical,93,530,0.731,0.447,0.284,52.9,1.42,23.7
16,Physical Phenomenon,72,278,0.944,0.665,0.279,91.2,1.65,15.3
13,Body of Water,55,146,0.673,0.411,0.262,62.2,1.64,16.4
9,Location,127,489,0.740,0.501,0.239,59.6,1.65,24.4
20,Ecosystem,56,89,0.661,0.449,0.211,64.9,1.71,16.1
6,Method,107,536,0.907,0.739,0.168,71.1,1.87,16.8
0,Organism,110,403,0.718,0.556,0.162,57.0,1.69,49.1
21,Natural Disaster,48,52,0.812,0.673,0.139,71.8,1.48,8.3
15,Quantity,306,1321,0.794,0.665,0.129,66.7,2.11,24.2
